This file is for analyzing the EHT data. All analysis is finished and saved in the folder. If you want to rerun the analysis, please set RERUN_ANALYSIS = True.

What is needed:

- `result/230_{day}.pkl` image of different hyperparam combinations

What is generated:

- `{day}_all_data_mean.npy` averaged IQU of all combinations
- `{day}_all_data.npy` IQU of all combinations
- `{day}.npz` theta_mean,theta_std,P_mean...
- `{day}_EVPA_1d.npy` evpa,q,u of all combinations
- `{day}_1d.npz` 1d theta_mean,theta_std,P_mean...

In [1]:
RERUN_ANALYSIS = False

In [2]:
import pickle
import numpy as np
import scipy.stats as stats
import os
import numpy as np
from copy import deepcopy
from scipy import optimize
import scipy
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.libanaly import Plot_Background,plot_quivers,CircleCalc

below are some functions.

In [3]:
def get_reloc_offset(dataimage):
    """
    This function receives a dataimage and returns the offset of the center of the image, so that you can relocate the blackhole to the center.
    """

    shape=dataimage.shape[0]
    xgrid = np.linspace(-16.3434,16.3434,shape,endpoint=False)
    ygrid = np.linspace(-16.3434,16.3434,shape,endpoint=False)
    diff = np.diff(xgrid)[0]/2.
    xgrid += diff
    ygrid += diff
    xmesh,ymesh = np.meshgrid(xgrid,ygrid,indexing='ij')

    iI = scipy.interpolate.RectBivariateSpline(xmesh[:,0],ymesh[0,:],dataimage[:,:,0])

    Rres = 10
    Rmax = 10
    Rmin = 0.2
    
    Pres = 10
    Pmax = 1 * 2*np.pi
    Pmin = 1./360. * 2*np.pi
    
    
    dr = Rmin/Rres
    dphi = Pmin/Pres

    
    r = np.arange(np.round(Rmax/Rmin)*Rres + 1)*dr
    phi = ( np.arange(np.round(Pmax/Pmin)*Pres  + 1) -Pres/2 )*dphi
    
    window = np.ones((Rres+1,Pres+1))
    window[0,:]  *= 0.5
    window[-1,:] *= 0.5
    window[:,0]  *= 0.5
    window[:,-1] *= 0.5
    
    
    
    window0 = np.ones(Pres+1)
    window0[0]  *= 0.5
    window0[-1] *= 0.5
    
    r_pk = np.zeros((17*17,len(phi)))
    r_pk_bar = np.zeros((17*17))
    sigma_r_bar = np.zeros((17*17))
    starting = int(shape/2-8)
    
    for x0 in range(17):
        for y0 in range(17):
            x_mesh = r[:,None]*np.cos(phi[None,:])+xmesh[:,0][x0+starting]
            y_mesh = r[:,None]*np.sin(phi[None,:])+ymesh[0,:][y0+starting]
            Ii = iI.ev(x_mesh,y_mesh) 
            r_pk[x0*17+y0] = np.argmax(Ii,axis=0)
            r_pk_bar[x0*17+y0] = np.mean(r_pk[x0*17+y0])
            sigma_r_bar[x0*17+y0] = np.std(r_pk[x0*17+y0])
    central = np.argmin(sigma_r_bar/r_pk_bar)
    xoff = int(central//17+starting-shape/2-1)
    yoff = int(central%17+starting-shape/2-1)
    
    return xoff,yoff

In [4]:
def gaussian(x,*param):
    mean = param[0]
    sigma = param[1]
    amplitude = param[2]
    return amplitude/np.sqrt(2.*np.pi*sigma*sigma)*np.exp(-np.power(x-mean,2.)/2./sigma/sigma)

    
def normalize_evpa(evpaarr):
    return ((evpaarr + 0.5 * np.pi) % np.pi) - 0.5 * np.pi
    

def vonmises_mean(d1evpaarr):
    # d1evpaarr is from -0.5pi to 0.5pi
    kappa_fit, mu_fit ,_= stats.vonmises.fit(d1evpaarr*2)
    return mu_fit/2
    

def fit_gaussian2(d1evpaarr,bins_i):
    counts,bins = np.histogram(d1evpaarr,bins=10*bins_i) #10*(4)
    #counts,bins =re_loc(counts,bins)
    xarray = np.array([(bins[i]+bins[i+1])/2 for i in range(len(bins)-1)])
    popt,pcov = optimize.curve_fit(gaussian,xarray,counts,p0=[xarray[np.argmax(counts)],5,d1evpaarr.shape[0]],maxfev=500000,gtol=1e-10)
    if abs(pcov[0][0])>abs(popt[0]) or abs(pcov[1][1])>abs(popt[1]):
        print(f'badfit')
        return np.array([np.nan,np.nan,np.nan])
    else:
        return popt
    
    
##########################################################################################
# These functions are not used in the code, if you are interested in them, try it :)######
##########################################################################################

def fit_gaussian_calc(d1evpaarr):
    # d1evpaarr is from -0.5pi to 0.5pi
    kappa_fit, mu_fit ,_= stats.vonmises.fit(d1evpaarr*2)
    sigma=np.sqrt(1/kappa_fit)/2
    mu_fit = mu_fit/2
    t=deepcopy(d1evpaarr)
    t[t<-0.5*np.pi+mu_fit]+=np.pi
    t[t>0.5*np.pi+mu_fit]-=np.pi
    # use stats.norm.fit
    mu_fit,sigma_fit = stats.norm.fit(t)
    return np.array([mu_fit,sigma_fit])

def fit_vonmise(d1evpaarr):
    # d1evpaarr is from -0.5pi to pi
    kappa_fit, mu_fit ,_= stats.vonmises.fit(d1evpaarr*2)
    sigma=np.sqrt(1/kappa_fit)/2
    mu_fit = mu_fit/2
    t=deepcopy(d1evpaarr)
    t[t<-0.5*np.pi+mu_fit]+=np.pi
    t[t>0.5*np.pi+mu_fit]-=np.pi
    s,p=stats.kstest(t,'norm',args=(mu_fit,sigma))
    if p>0.001 or True:
        return np.array([mu_fit,sigma])
    else:
        print(f"badfit,p={p},mu={mu_fit},sigma={sigma}")
        return np.array([np.nan,np.nan])

2D EHT Image: relocate images and Fit it

In [5]:
# offset_day is the result of get_reloc_offset function. If you think this block is time wasing, please set USING_CALCED_OFFSET = True and directly use the offset_day value.
USING_CALCED_OFFSET = False
offset_day={95:(2,0),96:(1,1),100:(-4,4),101:(-1,1)}

len_top = 1572
for day in [95,96,100,101] if RERUN_ANALYSIS else []:
    print(day)
    with open(f"result/230_{day}.pkl",'rb') as file1:
        data = pickle.loads(file1.read())
    chi2averhist = [pic['chi2'] for pic in data['meta_data']]
    data['meta_data'].sort(key= lambda k:(k.get('chi2',0)))
    chi2aver = deepcopy(data['meta_data'])

    theta_mean = np.zeros((64,64))
    theta_std = np.zeros((64,64))
    theta_all = np.zeros((64,64))
    theta_data = np.zeros((len_top,64,64))
    I_data = np.zeros((len_top,64,64))
    Q_data = np.zeros((len_top,64,64))
    U_data = np.zeros((len_top,64,64))
    All_data = np.zeros((len_top,64,64,4))


    All_data[:,:,:,0]=np.transpose(data['I_data'][::-1][:,:,:len_top],(2,0,1))
    All_data[:,:,:,1]=np.transpose(data['Q_data'][::-1][:,:,:len_top],(2,0,1))
    All_data[:,:,:,2]=np.transpose(data['U_data'][::-1][:,:,:len_top],(2,0,1))
    # These rotations are used to make the data in the same direction as the GRMHD data. 
    # If you use our Plot_Background function, the matrix after rotate will output the correct image.
    All_data[:,:,:,0]=np.rot90(All_data[:,:,:,0],k=-1,axes=(1,2))
    All_data[:,:,:,1]=-np.rot90(All_data[:,:,:,1],k=-1,axes=(1,2))
    All_data[:,:,:,2]=-np.rot90(All_data[:,:,:,2],k=-1,axes=(1,2))
    theta_data=0.5*np.arctan2(All_data[:,:,:,2],All_data[:,:,:,1]) # rad
    # move theta data to put vonmises mean at the center of theta range, that helps gaussian fit
    theta_vonmises_mean=np.apply_along_axis(vonmises_mean,0,theta_data)
    theta_data=CircleCalc.circle_sub_evpa(theta_data,theta_vonmises_mean)+theta_vonmises_mean


    for bins_i in [10]: #,6,4,14]: # bins_i=10 gives the best fit result (will get the most good fit points)
        res_arr=np.apply_along_axis(fit_gaussian2,0,theta_data,bins_i)
        theta_mean=res_arr[0,:,:]
        # although theta_vonmises_mean is (-0.5pi, 0.5pi), the theta_mean may be out of this range, so we need to normalize it to (-0.5pi, 0.5pi).
        theta_mean=normalize_evpa(theta_mean)
        theta_std=np.abs(res_arr[1,:,:])

    

    All_data_mean = np.mean(All_data,axis=0)
    if USING_CALCED_OFFSET:
        xoff,yoff = offset_day[day]
    else:
        xoff,yoff  = get_reloc_offset(All_data_mean)
    print(f'xoff:{xoff},yoff:{yoff}')

    # This part relocates theta_mean, theta_std, All_data_mean and All_data, moves the blackhole to the center of the image.
    bigset = np.zeros((2,64+16,64+16))
    bigset2=np.zeros((64+16,64+16,4))
    bigset3=np.zeros((len_top,64+16,64+16,4))

    bigset[0, 9:64+9, 9:64+9] = theta_mean
    bigset[1, 9:64+9, 9:64+9] = theta_std
    bigset2[9:64+9, 9:64+9] = All_data_mean
    bigset3[:,9:64+9, 9:64+9] = All_data
    
    theta_mean[:] = bigset[0, 9 + xoff:64+9 + xoff, 9 + yoff:64+9 + yoff]
    theta_std[:] = bigset[1, 9 + xoff:64+9 + xoff, 9 + yoff:64+9 + yoff]
    All_data_mean[:]=bigset2[9 + xoff:64+9 + xoff, 9 + yoff:64+9 + yoff]
    All_data[:]=bigset3[:,9 + xoff:64+9 + xoff, 9 + yoff:64+9 + yoff]
    
    np.save(f'{day}_all_data_mean',All_data_mean)
    np.save(f'{day}_all_data',All_data)

    P_mean = np.sqrt(All_data_mean[:,:,1]**2+All_data_mean[:,:,2]**2)
    np.savez(f'{day}',theta_mean,theta_std,P_mean,All_data_mean[:,:,0])
    

1D EHT Image: calculate the 1D EVPA (Integrate the radius), and Fit it

In [ ]:
def Create_EVPA_Dataset(image1):
    """
    This Function calculates 1D EVPA from 2D image.
    """

    xgrid = np.linspace(-16.3434, 16.3434, 64, endpoint=False)
    ygrid = np.linspace(-16.3434, 16.3434, 64, endpoint=False)
    diff = np.diff(xgrid)[0] / 2.0
    xgrid += diff
    ygrid += diff
    xmesh, ymesh = np.meshgrid(xgrid, ygrid, indexing="ij")

    iQ = scipy.interpolate.RectBivariateSpline(
        xmesh[:, 0], ymesh[0, :], image1[:,:,1]
    )
    iU = scipy.interpolate.RectBivariateSpline(
        xmesh[:, 0], ymesh[0, :], image1[:,:,2]
    )
    iI = scipy.interpolate.RectBivariateSpline(
        xmesh[:, 0], ymesh[0, :], image1[:,:,0]
    )

    Rres = 10
    Rmax = 16.343
    Rmin = 0.2

    Pres = 10
    Pmax = 1 * 2 * np.pi
    Pmin = 1.0 / 360.0 * 2 * np.pi

    dr = Rmin / Rres
    dphi = Pmin / Pres

    r = np.arange(np.round(Rmax / Rmin) * Rres + 1) * dr
    phi = (np.arange(np.round(Pmax / Pmin) * Pres + 1) - Pres / 2) * dphi

    window = np.ones((Rres + 1, Pres + 1))
    window[0, :] *= 0.5
    window[-1, :] *= 0.5
    window[:, 0] *= 0.5
    window[:, -1] *= 0.5

    window0 = np.ones(Pres + 1)
    window0[0] *= 0.5
    window0[-1] *= 0.5

    x_mesh = r[:, None] * np.cos(phi[None, :])
    y_mesh = r[:, None] * np.sin(phi[None, :])

    Ii = iI.ev(x_mesh, y_mesh)
    Qi = iQ.ev(x_mesh, y_mesh)
    Ui = iU.ev(x_mesh, y_mesh)

    integrand_Q = r[:, None] * Qi * Ii * dr * dphi
    integrand_U = r[:, None] * Ui * Ii * dr * dphi

    # fast integration, for each point there are 6*6 blocks to be integrated, the border overlaps.
    # fast?
    Rn = round(Rmax / Rmin)
    Pn = round(Pmax / Pmin)

    integrated_Q_phi = np.array(
        [
            (integrand_Q[:, Pres * j : Pres * (j + 1) + 1] * window0).sum()
            for j in range(Pn)
        ]
    )
    integrated_U_phi = np.array(
        [
            (integrand_U[:, Pres * j : Pres * (j + 1) + 1] * window0).sum()
            for j in range(Pn)
        ]
    )

    integrated_Q_phi_10deg = np.array(
        [
            np.sum([integrated_Q_phi[(i + k - 1) % Pn] for k in range(3)])
            for i in range(Pn)
        ]
    )
    integrated_U_phi_10deg = np.array(
        [
            np.sum([integrated_U_phi[(i + k - 1) % Pn] for k in range(3)])
            for i in range(Pn)
        ]
    )

    evpa_phi_10deg = (
        (180.0 / np.pi)
        * 0.5
        * np.arctan2(integrated_U_phi_10deg, integrated_Q_phi_10deg)
    )
    return evpa_phi_10deg, integrated_Q_phi_10deg, integrated_U_phi_10deg

In [ ]:
for day in [95,96,100,101] if RERUN_ANALYSIS else []:
    print(day)
    All_data=np.load(f'{day}_all_data.npy')
    EVPA_1d=np.zeros((All_data.shape[0],360))
    for i in range(All_data.shape[0]):
        EVPA_1d[i],_,_=Create_EVPA_Dataset(All_data[i])/180*np.pi
    np.save(f'{day}_EVPA_1d',EVPA_1d)

In [8]:
for day in [95,96,100,101] if RERUN_ANALYSIS else []:
    print(day)
    EVPA_1d=np.load(f'{day}_EVPA_1d.npy')
    EVPA_vonmises_mean=np.apply_along_axis(vonmises_mean,0,EVPA_1d)
    EVPA_1d=CircleCalc.circle_sub_evpa(EVPA_1d,EVPA_vonmises_mean)+EVPA_vonmises_mean

    for bins_i in [10]: #,6,4,14]:
        res_arr=np.apply_along_axis(fit_gaussian2,0,EVPA_1d,bins_i)
        theta_mean=res_arr[0]
        theta_mean=normalize_evpa(theta_mean)
        theta_std=np.abs(res_arr[1])
        np.savez(f'{day}_1d',theta_mean,theta_std)
    